# Fish SimCLR Clean Pipeline

This notebook is a cleaned, safer version of `fish_simCLR.ipynb`.

Main fixes:
- Uses the HPC dataset root: `/cluster/pixstor/hdtg3-lab/jlc3q/trout/new`.
- Builds the image/label/demo table directly from `labeling.xlsx`, `demo.xlsx`, and image folders.
- Adds data-conflict checks before training.
- Avoids CUDA-only loading and high-worker defaults that often crash notebooks.
- Uses `state_dict` checkpoints instead of pickling entire model objects.
- Makes SimCLR NT-Xent loss work with the actual batch size, so the last batch does not break.

## 1. Setup

Run this section first. If `torch` or `torchvision` is missing, install them in the active Jupyter kernel before running the training cells.

In [ ]:
from __future__ import annotations

import os
import re
import random
from pathlib import Path

import numpy as np
import pandas as pd

ROOT_DIR = Path("/cluster/pixstor/hdtg3-lab/jlc3q/trout/new")
CODE_DIR = Path("/home/jlc3q/Trout/TroutCNN/code")
DEMO_XLSX = CODE_DIR / "demo.xlsx"
LABEL_XLSX = CODE_DIR / "labeling.xlsx"
OUTPUT_DIR = CODE_DIR / "simclr_clean_outputs"
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 100

def set_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed(seed)
            torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    except Exception:
        pass

def normalize_text(value):
    if pd.isna(value):
        return np.nan
    return " ".join(str(value).strip().split())

set_seed()
print("ROOT_DIR:", ROOT_DIR)
print("demo.xlsx exists:", DEMO_XLSX.exists())
print("labeling.xlsx exists:", LABEL_XLSX.exists())

## 2. Build Master Table

In [ ]:
def build_image_index(root: Path) -> pd.DataFrame:
    image_paths = sorted(
        p for p in root.glob("*/*/*.png")
        if p.is_file() and not p.name.startswith("._")
    )
    pattern = re.compile(r"^([a-z]+)(\d+)_(\d+)_(\d+)\.png$", re.IGNORECASE)
    rows, bad = [], []

    for path in image_paths:
        match = pattern.match(path.name)
        if not match:
            bad.append(str(path))
            continue
        river, point, fish_id, image_idx = match.groups()
        river = river.lower()
        point = int(point)
        fish_id = int(fish_id)
        image_idx = int(image_idx)
        fish_key = f"{river}{point}_{fish_id:03d}"
        scale_id = f"{fish_key}_{image_idx:02d}"
        rows.append({
            "path": str(path),
            "file": path.name,
            "scale_id": scale_id,
            "fish_key": fish_key,
            "river": river.upper(),
            "point": point,
            "fish_id": fish_id,
            "image_idx": image_idx,
        })

    df = pd.DataFrame(rows)
    print(f"Parsed images: {len(df):,}")
    print(f"Skipped bad filenames: {len(bad):,}")
    if bad[:5]:
        print("Bad filename examples:", bad[:5])
    return df

images_df = build_image_index(ROOT_DIR)
images_df.head()

In [ ]:
def clean_label(age_est, obs):
    if pd.notna(age_est):
        numeric = pd.to_numeric(age_est, errors="coerce")
        if pd.notna(numeric):
            return int(numeric)

    text = " ".join(
        normalize_text(x).lower()
        for x in [age_est, obs]
        if pd.notna(x)
    )
    if any(term in text for term in ["not readable", "regenerated", "broken"]):
        return 6
    return np.nan

labels_df = pd.read_excel(LABEL_XLSX, sheet_name="mizzou_allscales")
labels_df["scale_id"] = labels_df["scale_id"].map(lambda x: str(x).strip().lower())
labels_df["label"] = labels_df.apply(lambda r: clean_label(r["age_est"], r["obs"]), axis=1)
labels_df["label_source"] = np.where(labels_df["label"].notna(), "expert", "unlabeled")

demo_df = pd.read_excel(DEMO_XLSX, sheet_name="all")
demo_df = demo_df.rename(columns={"id": "fish_key", "lenght (mm)": "length_mm", "weight (g)": "weight_g"})
demo_df["fish_key"] = demo_df["fish_key"].map(lambda x: str(x).strip().lower())

master_df = (
    images_df
    .merge(labels_df[["scale_id", "sampling_date", "age_est", "obs", "label", "label_source"]],
           on="scale_id", how="left")
    .merge(demo_df[["fish_key", "id_old", "length_mm", "weight_g"]],
           on="fish_key", how="left")
)

master_df["split"] = np.where(master_df["label"].notna(), "labeled", "unlabeled")
master_df["known_bad"] = np.where(master_df["label"].eq(6), 1, 0)

print("master_df:", master_df.shape)
print(master_df["split"].value_counts(dropna=False))
print(master_df["label"].value_counts(dropna=False).sort_index())
master_df.head()

## 3. Data Checks and Conflict Audit

These cells are intentionally near the top. Do not train until the counts and conflicts look reasonable.

In [ ]:
def audit_master(master: pd.DataFrame, labels: pd.DataFrame, demo: pd.DataFrame) -> dict:
    duplicate_scale_ids = labels[labels["scale_id"].duplicated(keep=False)].sort_values("scale_id")
    images_without_labels = master[master["age_est"].isna() & master["obs"].isna()]
    images_without_demo = master[master["length_mm"].isna() | master["weight_g"].isna()]
    demo_without_images = demo.loc[~demo["fish_key"].isin(master["fish_key"])]

    numeric_labeled = master[master["label"].between(0, 5, inclusive="both")]
    label_sets = (
        numeric_labeled
        .groupby(["river", "point", "fish_id"])["label"]
        .agg(lambda s: sorted(set(int(x) for x in s.dropna())))
        .reset_index(name="labels")
    )
    fish_label_conflicts = label_sets[label_sets["labels"].map(len) > 1]

    summary = {
        "images": len(master),
        "labeled_rows": int(master["label"].notna().sum()),
        "unlabeled_rows": int(master["label"].isna().sum()),
        "duplicate_scale_id_rows": len(duplicate_scale_ids),
        "images_without_label_rows": len(images_without_labels),
        "images_without_demo_rows": len(images_without_demo),
        "demo_fish_without_images": len(demo_without_images),
        "fish_age_conflict_groups": len(fish_label_conflicts),
    }
    return summary, duplicate_scale_ids, images_without_labels, images_without_demo, demo_without_images, fish_label_conflicts

audit_summary, duplicate_scale_ids, images_without_labels, images_without_demo, demo_without_images, fish_label_conflicts = audit_master(master_df, labels_df, demo_df)
audit_summary

In [ ]:
display(fish_label_conflicts.head(30))
display(images_without_demo.head(10)[["scale_id", "fish_key", "path"]])
display(duplicate_scale_ids.head(10))

audit_summary_path = OUTPUT_DIR / "data_audit_summary.csv"
pd.DataFrame([audit_summary]).to_csv(audit_summary_path, index=False)
fish_label_conflicts.to_csv(OUTPUT_DIR / "fish_label_conflicts.csv", index=False)
print("Saved audit files under:", OUTPUT_DIR)

## 4. Torch Setup

In [ ]:
try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import Dataset, DataLoader
    import torchvision.transforms as transforms
    import torchvision.models as models
    from PIL import Image
    from tqdm.auto import tqdm
except ImportError as exc:
    raise ImportError(
        "This notebook needs torch, torchvision, pillow, and tqdm in the active Jupyter kernel. "
        "Install them, then restart the kernel."
    ) from exc

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_CUDA = device.type == "cuda"
NUM_WORKERS = min(4, os.cpu_count() or 1)
PIN_MEMORY = USE_CUDA

print("device:", device)
print("num_workers:", NUM_WORKERS)

In [ ]:
def make_resnet18_backbone(pretrained: bool = True) -> nn.Module:
    try:
        weights = models.ResNet18_Weights.DEFAULT if pretrained else None
        model = models.resnet18(weights=weights)
    except AttributeError:
        model = models.resnet18(pretrained=pretrained)
    model.fc = nn.Identity()
    return model

class SimCLRDataset(Dataset):
    def __init__(self, dataframe: pd.DataFrame, transform):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["path"]).convert("RGB")
        return self.transform(img), self.transform(img)

class ImageLabelDataset(Dataset):
    def __init__(self, X: pd.DataFrame, y, transform):
        self.X = X.reset_index(drop=True)
        self.y = pd.Series(y).reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        img = Image.open(self.X.iloc[idx]["path"]).convert("RGB")
        return self.transform(img), int(self.y.iloc[idx])

class ImageOnlyDataset(Dataset):
    def __init__(self, dataframe: pd.DataFrame, transform):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img = Image.open(self.df.iloc[idx]["path"]).convert("RGB")
        return self.transform(img)

class NTXentLoss(nn.Module):
    def __init__(self, temperature: float = 0.2):
        super().__init__()
        self.temperature = temperature
        self.criterion = nn.CrossEntropyLoss(reduction="sum")

    def forward(self, z1, z2):
        batch_size = z1.size(0)
        z = torch.cat((z1, z2), dim=0)
        sim = F.cosine_similarity(z.unsqueeze(1), z.unsqueeze(0), dim=2)

        positive = torch.cat([
            torch.diag(sim, batch_size),
            torch.diag(sim, -batch_size),
        ], dim=0).unsqueeze(1)

        mask = torch.ones((2 * batch_size, 2 * batch_size), dtype=torch.bool, device=z.device)
        mask.fill_diagonal_(False)
        for i in range(batch_size):
            mask[i, batch_size + i] = False
            mask[batch_size + i, i] = False

        negative = sim[mask].view(2 * batch_size, -1)
        logits = torch.cat([positive, negative], dim=1) / self.temperature
        labels = torch.zeros(2 * batch_size, dtype=torch.long, device=z.device)
        return self.criterion(logits, labels) / (2 * batch_size)

## 5. Stage A: Bad/Regenerated Classifier

In [ ]:
from sklearn.model_selection import train_test_split

labeled_df = master_df[master_df["label"].notna()].reset_index(drop=True)
unlabeled_df = master_df[master_df["label"].isna()].reset_index(drop=True)

labeled_df["known_bad"] = np.where(labeled_df["label"].eq(6), 1, 0)

train_bad, test_bad = train_test_split(
    labeled_df,
    test_size=0.2,
    stratify=labeled_df["known_bad"],
    random_state=SEED,
)
train_bad = train_bad.reset_index(drop=True)
test_bad = test_bad.reset_index(drop=True)

print("train_bad:", train_bad.shape, train_bad["known_bad"].value_counts().to_dict())
print("test_bad :", test_bad.shape, test_bad["known_bad"].value_counts().to_dict())

In [ ]:
simclr_transform = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.8, 0.8, 0.8, 0.2),
    transforms.RandomGrayscale(p=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [ ]:
def train_simclr(dataframe, epochs=10, batch_size=128, lr=3e-4, checkpoint_name="simclr_backbone.pt"):
    set_seed()
    dataset = SimCLRDataset(dataframe, simclr_transform)
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        drop_last=False,
    )

    backbone = make_resnet18_backbone(pretrained=True).to(device)
    projection_head = nn.Sequential(
        nn.Linear(512, 128),
        nn.ReLU(),
        nn.Linear(128, 128),
    ).to(device)

    optimizer = torch.optim.Adam(list(backbone.parameters()) + list(projection_head.parameters()), lr=lr)
    criterion = NTXentLoss(temperature=0.2)

    backbone.train()
    projection_head.train()
    for epoch in range(epochs):
        losses = []
        for x1, x2 in tqdm(loader, desc=f"SimCLR {epoch + 1}/{epochs}"):
            x1, x2 = x1.to(device), x2.to(device)
            z1 = projection_head(backbone(x1))
            z2 = projection_head(backbone(x2))
            loss = criterion(z1, z2)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            losses.append(float(loss.detach().cpu()))
        print(f"epoch={epoch + 1} loss={np.mean(losses):.4f}")

    ckpt_path = CHECKPOINT_DIR / checkpoint_name
    torch.save({"backbone_state_dict": backbone.state_dict()}, ckpt_path)
    print("saved:", ckpt_path)
    return backbone

# Long-running cell. Uncomment to train.
# bad_backbone = train_simclr(train_bad, epochs=10, checkpoint_name="bad_stage_simclr_backbone.pt")

In [ ]:
def load_backbone(checkpoint_name: str) -> nn.Module:
    backbone = make_resnet18_backbone(pretrained=False).to(device)
    ckpt_path = CHECKPOINT_DIR / checkpoint_name
    checkpoint = torch.load(ckpt_path, map_location=device)
    backbone.load_state_dict(checkpoint["backbone_state_dict"])
    backbone.eval()
    return backbone

# If you trained above, use bad_backbone directly.
# bad_backbone = load_backbone("bad_stage_simclr_backbone.pt")

In [ ]:
def train_classifier(backbone, train_df, test_df, target_col, epochs=10, batch_size=64, lr=1e-4):
    set_seed()
    train_dataset = ImageLabelDataset(train_df, train_df[target_col], eval_transform)
    test_dataset = ImageLabelDataset(test_df, test_df[target_col], eval_transform)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

    num_classes = int(pd.concat([train_df[target_col], test_df[target_col]]).nunique())
    classifier_head = nn.Sequential(
        nn.Linear(512, 128),
        nn.ReLU(),
        nn.Linear(128, num_classes),
    ).to(device)
    model = nn.Sequential(backbone, classifier_head).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    model.train()
    for epoch in range(epochs):
        losses = []
        for imgs, labels in tqdm(train_loader, desc=f"Classifier {epoch + 1}/{epochs}"):
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            losses.append(float(loss.detach().cpu()))
        print(f"epoch={epoch + 1} loss={np.mean(losses):.4f}")

    return model, classifier_head, test_loader

# Long-running cell. Uncomment after `bad_backbone` is available.
# bad_model, bad_head, bad_test_loader = train_classifier(bad_backbone, train_bad, test_bad, "known_bad", epochs=10)

In [ ]:
from sklearn.metrics import classification_report

def evaluate_classifier(model, loader, class_names=None):
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            outputs = model(imgs)
            preds = outputs.argmax(dim=1).cpu().numpy()
            y_pred.extend(preds)
            y_true.extend(labels.numpy())
    print(classification_report(y_true, y_pred, target_names=class_names, digits=4))
    return np.array(y_true), np.array(y_pred)

# evaluate_classifier(bad_model, bad_test_loader, class_names=["good", "bad_or_regenerated"])

## 6. Label Extension Without Silent Conflicts

In [ ]:
def extend_labels_by_fish_group(df: pd.DataFrame):
    combined = df.copy()
    source_label_col = "extended_label"
    combined[source_label_col] = combined["label"]

    expert_numeric = combined[combined["label"].between(0, 5, inclusive="both")]
    counts = (
        expert_numeric
        .groupby(["river", "point", "fish_id"])["label"]
        .value_counts()
        .rename("count")
        .reset_index()
    )

    fill_map = {}
    conflicts = []
    for key, part in counts.groupby(["river", "point", "fish_id"]):
        max_count = part["count"].max()
        top = sorted(part.loc[part["count"].eq(max_count), "label"].astype(int).tolist())
        if len(top) == 1:
            fill_map[key] = top[0]
        else:
            conflicts.append({"river": key[0], "point": key[1], "fish_id": key[2], "candidate_labels": top})

    target = combined[source_label_col].isna()
    for (river, point, fish_id), label in fill_map.items():
        mask = (
            target
            & combined["river"].eq(river)
            & combined["point"].eq(point)
            & combined["fish_id"].eq(fish_id)
        )
        combined.loc[mask, source_label_col] = label

    conflict_df = pd.DataFrame(conflicts)
    return combined, conflict_df

combined_df, extension_conflicts = extend_labels_by_fish_group(master_df)
print("Before:", master_df["label"].notna().sum(), "After:", combined_df["extended_label"].notna().sum())
display(extension_conflicts.head(30))
combined_df.to_csv(OUTPUT_DIR / "master_with_extended_labels.csv", index=False)
extension_conflicts.to_csv(OUTPUT_DIR / "extension_conflicts.csv", index=False)

## 7. Stage B: Age Classifier 0-5

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

age_df = combined_df[combined_df["extended_label"].between(0, 5, inclusive="both")].copy()
age_df["age_class"] = age_df["extended_label"].astype(int)
age_df["group_id"] = (
    age_df["river"].astype(str) + "_" +
    age_df["point"].astype(str) + "_" +
    age_df["fish_id"].astype(str)
)

X = age_df.drop(columns=["age_class"])
y = age_df["age_class"]
groups = age_df["group_id"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
train_idx, test_idx = next(gss.split(X, y, groups))
X_train, X_test = X.iloc[train_idx].reset_index(drop=True), X.iloc[test_idx].reset_index(drop=True)
y_train, y_test = y.iloc[train_idx].reset_index(drop=True), y.iloc[test_idx].reset_index(drop=True)

train_age = X_train.copy()
train_age["age_class"] = y_train
test_age = X_test.copy()
test_age["age_class"] = y_test

print("train_age:", train_age.shape, y_train.value_counts().sort_index().to_dict())
print("test_age :", test_age.shape, y_test.value_counts().sort_index().to_dict())

In [ ]:
# Long-running cells. Uncomment when ready.
# age_backbone = train_simclr(train_age, epochs=10, checkpoint_name="age_stage_simclr_backbone.pt")
# age_model, age_head, age_test_loader = train_classifier(age_backbone, train_age, test_age, "age_class", epochs=10)
# evaluate_classifier(age_model, age_test_loader, class_names=[str(i) for i in range(6)])

## 8. Save/Load Full Supervised Checkpoints

In [ ]:
def save_supervised_checkpoint(backbone, classifier_head, class_names, checkpoint_name):
    path = CHECKPOINT_DIR / checkpoint_name
    torch.save({
        "backbone_state_dict": backbone.state_dict(),
        "head_state_dict": classifier_head.state_dict(),
        "class_names": list(class_names),
    }, path)
    print("saved:", path)

def load_supervised_checkpoint(checkpoint_name, num_classes):
    backbone = make_resnet18_backbone(pretrained=False).to(device)
    head = nn.Sequential(
        nn.Linear(512, 128),
        nn.ReLU(),
        nn.Linear(128, num_classes),
    ).to(device)
    checkpoint = torch.load(CHECKPOINT_DIR / checkpoint_name, map_location=device)
    backbone.load_state_dict(checkpoint["backbone_state_dict"])
    head.load_state_dict(checkpoint["head_state_dict"])
    model = nn.Sequential(backbone, head).to(device)
    model.eval()
    return model, checkpoint.get("class_names")

# save_supervised_checkpoint(age_backbone, age_head, class_names=[str(i) for i in range(6)], checkpoint_name="age_stage_backbone_head.pt")

## 9. Notes on Original Notebook Failure Points

The original notebook likely fails or produces unstable results in these places:

- `ROOT_DIR = Path("data/fish/")` does not match the external-drive dataset layout.
- `torch.load(..., map_location='cuda')` fails on CPU-only machines. This version uses `map_location=device`.
- `num_workers=10` or `12` can crash or hang notebooks on macOS/Jupyter. This version caps workers at 4.
- Original `NTXentLoss(batch_size=128)` assumes every batch is exactly 128. This version computes masks from the actual batch size.
- `models.resnet18(pretrained=True)` is deprecated in newer torchvision. This version supports both old and new APIs.
- Saving/loading entire model objects is brittle. This version saves `state_dict`s.
- Label propagation now exports conflict cases instead of silently skipping or hiding them.